# Capstone — 3 seeds x 20k steps, joint modular + induction (GPU)

Issue #170. The planned `configs/capstone.yaml` run: joint training at d_model=256, 4 layers, 20,000 optimizer steps per seed, full mechanistic instrumentation (Fourier progress measures, K-composition, checkpointing every 1000 steps).

**Resume is the point.** Kaggle sessions cap around 9h and Colab disconnects; the run is built to be *paused*, not lost. Each seed writes `checkpoints/exp6_capstone_seed<N>_step<S>.pt` every 1000 steps (checkpointing is armed by `--save-model`). The run cell computes the last common checkpoint step across seeds and passes it as `--resume`, so re-running the cell after any interruption continues exactly where the session stopped.

**W&B is optional.** Set `WANDB_API_KEY` below (or add it as an environment secret) and the run logs to W&B; without it the run proceeds identically — the manifests are the record, not the dashboard.

**Before running:** enable a GPU (Kaggle: Settings -> Accelerator -> GPU, Internet On; Colab: Runtime -> Change runtime type -> GPU).

In [ ]:
BRANCH = "main"  # a pushed branch or tag — pin it so the manifest's git_sha is a real commit
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

In [ ]:
!pip install -q uv
!uv sync --frozen
!uv run python -c "import torch; assert torch.cuda.is_available(), 'no GPU — enable one before running'; print(torch.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# Optional W&B. Leave the token empty to run without it — the manifests are
# the record either way.
import os

WANDB_API_KEY = ""  # paste a key here for a W&B-tracked run (private notebook)
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    !uv pip install -q wandb
    print("W&B enabled.")
else:
    print("W&B disabled (no key) — running manifest-only.")

In [ ]:
# The 20k-step run. Resumable: after a disconnect, re-run THIS cell — it
# finds the last checkpoint step every seed reached and resumes from there.
import glob
import os
import re

CKPT_DIR = "checkpoints"
SEEDS = [0, 1, 2]

def last_common_step() -> int:
    """Highest step at which EVERY seed has a checkpoint (0 = start fresh)."""
    per_seed = []
    for seed in SEEDS:
        steps = [int(m.group(1)) for p in glob.glob(os.path.join(CKPT_DIR, f"exp6_capstone_seed{seed}_step*.pt"))
                 if (m := re.search(r"_step(\d+)\.pt$", p))]
        per_seed.append(max(steps) if steps else 0)
    return min(per_seed)

os.makedirs(CKPT_DIR, exist_ok=True)
last = last_common_step()
print(f"resuming from step {last}" if last else "fresh run")

if WANDB_API_KEY:
    !uv run python -m src.experiments.exp6_capstone --seeds 0,1,2 --steps 20000 --warmup-steps 1000 --checkpoint-every 1000 --save-model --resume $last --wandb --wandb-project from-gradient-to-transformer-capstone --manifest-path results/capstone_20k.json
else:
    !uv run python -m src.experiments.exp6_capstone --seeds 0,1,2 --steps 20000 --warmup-steps 1000 --checkpoint-every 1000 --save-model --resume $last --manifest-path results/capstone_20k.json

In [ ]:
# Report the aggregate: modular vs induction (the MP-88 dissociation), Fourier
# density, and max K-composition — per seed and aggregated.
import json

m = json.load(open("results/capstone_20k.json"))
print("device:", m["device"], "| git:", m["git_sha"], "| dirty:", m["git_dirty"])
for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
    print(f"seed {seed}: modular {metrics['final_modular_acc']:.4f} | "
          f"induction {metrics['final_induction_acc']:.4f} | "
          f"k_99 {metrics['final_k_99']:.1f} | max K-comp {metrics['final_max_kcomp']:.4f}")
for k, v in m["aggregate"].items():
    print(f"{k:<22}{v['mean']:10.4f} +- {v['std']:.4f}  (n={int(v['n'])})")

In [ ]:
!zip -j capstone_20k.zip results/capstone_20k.json
try:
    from google.colab import files
    files.download("capstone_20k.zip")
except ImportError:
    print("Kaggle: the zip is in the notebook output (/kaggle/working).")

## After downloading

1. Unzip into a scratch dir, then pin: `python scripts/pin_remote_run.py --from <dir>`.
2. Write the run into `07_capstone/execution-log.md` (date, config, wall clock, outcome) — only runs that actually happened belong there.
3. Cite it in portfolio/RESULTS.md with a `<!-- manifest: results/capstone_20k.json -->` tag and re-run `make verify-claims`.
4. Note honestly: the pending `07_capstone/design.md` (issue #170's documentation half) is still local, non-GPU work — this run produces the evidence it describes.